In [ ]:
import xml.etree.ElementTree as ET

def extract_wikipedia_text(xml_path, max_pages=None):
    """Extracts plain text from <text> tags in the Wikipedia XML dump."""
    extracted_text = []
    page_count = 0

    for event, elem in ET.iterparse(xml_path, events=("start", "end")):
        if event == "end" and elem.tag.endswith("text"):
            text_content = elem.text
            if text_content:
                # Basic cleanup: remove things like [[link|display]], {{templates}}, and ==
                cleaned = (
                    text_content
                    .replace('\n', ' ')  # remove newlines
                    .replace('[[', '').replace(']]', '')
                )
                extracted_text.append(cleaned)

                page_count += 1
                if max_pages and page_count >= max_pages:
                    break
            elem.clear()

    return " ".join(extracted_text)

In [ ]:
!python -m wikiextractor.WikiExtractor "C:/Users/sang/OneDrive/Desktop/LLM_Marathi/simplewiki-latest-pages-articles.xml/simplewiki-latest-pages-articles.xml" -o extracted_simplewiki --json

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\sang\anaconda3\Lib\site-packages\wikiextractor\WikiExtractor.py", line 66, in <module>
    from .extract import Extractor, ignoreTag, define_template, acceptedNamespaces
  File "C:\Users\sang\anaconda3\Lib\site-packages\wikiextractor\extract.py", line 378, in <module>
    ExtLinkBracketedRegex = re.compile(
                            ^^^^^^^^^^^
  File "C:\Users\sang\anaconda3\Lib\re\__init__.py", line 227, in compile
    return _compile(pattern, flags)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\sang\anaconda3\Lib\re\__init__.py", line 294, in _compile
    p = _compiler.compile(pattern, flags)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\sang\anaconda3\Lib\re\_compiler.py", line 745, in compile
    p = _parser.parse(p, flags)
        ^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\sang\anaconda3\Lib\re\_parser.py"

In [ ]:
# Set the path to your Simple English Wikipedia XML file
wiki_path = r"S:\LLM_Marathi\simplewiki-latest-pages-articles.xml\simplewiki-latest-pages-articles.xml"

# Extract text (you can set max_pages to limit processing time)
simplewiki_text = extract_wikipedia_text(wiki_path, max_pages=1000)

# View sample
print("Sample text:\n", simplewiki_text[:1000])

Sample text:
 {{monththisyear|4}} '''April''' (Apr.) is the fourth month of the year in the Julian calendar|Julian and Gregorian calendars, and comes between March and May. It is one of four months to have 30 days.  April always begins on the same day of the week as July, and additionally, January in leap years. April always ends on the same day of the week as December.  == The Month == File:Colorful spring garden.jpg|thumb|180px|right|Spring (season)|Spring flowers in April in the Northern Hemisphere. April comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and November are later in the year.  April begins on the same day of the week as July every year and on the same day of the week as January in leap years. April ends on the same day of the week as December every year, as each other's last days are exactly 35 weeks (245 days) apart.  In common years, April starts on the sam

In [ ]:
# Assuming `simplewiki_text` already holds the extracted Wikipedia text
# CLEANING STEP
import re
clean_text = re.sub(r'[^\x00-\x7F]+', ' ', simplewiki_text)  # Remove non-ASCII chars
clean_text = re.sub(r'={2,}.*?={2,}', ' ', clean_text)       # Remove headers like "== History =="
clean_text = re.sub(r'\s+', ' ', clean_text).strip()         # Normalize whitespace

In [ ]:
# Load the full text first
# Save cleaned text (optional)
with open("simplewiki_cleaned.txt", "w", encoding="utf-8") as f:
    f.write(clean_text)


# Use only the first 100,000 characters (adjust as needed)
subset_text = clean_text[:500000]


In [ ]:
import torch
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F
import time
import os
# Configuration dictionary for the model
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

# Tokenizer initialization
tokenizer = tiktoken.get_encoding("gpt2")

# Dataset class for GPT Model
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

# Dataloader creation function
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

# GELU activation function (used in FeedForward layers)
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

# FeedForward network used in transformer block
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

# Layer normalization used in transformer blocks
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

# Multi-Head Attention (simplified version)
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, num_heads, dropout, qkv_bias=False):
        super().__init__()
        # Simplified placeholder, you can use a real implementation here
        pass

    def forward(self, x):
        return x  # Placeholder

# Transformer block, including MultiHeadAttention, FeedForward, and LayerNorm
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape: [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add shortcut

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add shortcut
        return x

# GPT Model implementation
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

# Text generation function (simple version)
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]  # Focus on the last time step
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

# Example usage for text generation
start_context = "Hello, I am"
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0)

model = GPTModel(GPT_CONFIG_124M)
model.eval()  # Disable dropout for evaluation

out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"]
)

decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

# Calculate total parameters and model size
total_params = sum(p.numel() for p in model.parameters())
total_size_bytes = total_params * 4  # assuming float32 (4 bytes per parameter)
total_size_mb = total_size_bytes / (1024 * 1024)
print(f"Total number of parameters: {total_params:,}")
print(f"Total size of the model: {total_size_mb:.2f} MB")

Hello, I am Chick starved,,,, defamation Orbital 164
Total number of parameters: 134,688,768
Total size of the model: 513.80 MB


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
import requests




# Save to file (Optional, if you need to save it)
with open("simplewiki_cleaned.txt", "w", encoding="utf-8") as f:
    f.write(subset_text)
    print("Total number of characters:", len(subset_text))
    print(subset_text[:99])  # Print the first 99 characters


# Define the dataset class
class GPTDatasetV1(torch.utils.data.Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # Use a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


# Load the tokenizer (You can use tiktoken for GPT-2 tokenizer)
tokenizer = tiktoken.get_encoding("gpt2")


# Create DataLoader function
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader


# Create the DataLoader
batch_size = 4
max_length = 256  # or any suitable value
stride = 128  # window slide size
train_dataloader = create_dataloader_v1(subset_text, batch_size, max_length, stride)


# Initialize model and move it to the appropriate device (e.g., GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPTModel(GPT_CONFIG_124M).to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# Training loop
num_epochs = 3  # or any suitable number of epochs

for epoch in range(num_epochs):
    start_train = time.time()
    model.train()  # Set model to training mode
    train_time_sec = round(time.time() - start_train, 2)
    total_loss = 0
    for batch_idx, (input_ids, target_ids) in enumerate(train_dataloader):
        input_ids, target_ids = input_ids.to(device), target_ids.to(device)

        # Forward pass
        logits = model(input_ids)

        # Compute loss
        loss = criterion(logits.view(-1, logits.size(-1)), target_ids.view(-1))

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Print loss every few batches
        if batch_idx % 10 == 0:
            print(f"Epoch {epoch + 1}/{num_epochs}, Batch {batch_idx}/{len(train_dataloader)}, Loss: {loss.item():.4f}")

    print(f"Epoch {epoch + 1} complete, Average Loss: {total_loss / len(train_dataloader):.4f}")

# Save the trained model (optional)
torch.save(model.state_dict(), "gpt_model.pth")

Total number of characters: 500000
{{monththisyear|4}} '''April''' (Apr.) is the fourth month of the year in the Julian calendar|Julia
Epoch 1/3, Batch 0/269, Loss: 11.0111
Epoch 1/3, Batch 10/269, Loss: 10.4651
Epoch 1/3, Batch 20/269, Loss: 9.4396
Epoch 1/3, Batch 30/269, Loss: 8.3200
Epoch 1/3, Batch 40/269, Loss: 7.7470
Epoch 1/3, Batch 50/269, Loss: 7.4677
Epoch 1/3, Batch 60/269, Loss: 7.3146
Epoch 1/3, Batch 70/269, Loss: 7.5816
Epoch 1/3, Batch 80/269, Loss: 6.9234
Epoch 1/3, Batch 90/269, Loss: 6.7295
Epoch 1/3, Batch 100/269, Loss: 6.9001
Epoch 1/3, Batch 110/269, Loss: 7.0765
Epoch 1/3, Batch 120/269, Loss: 6.8814
Epoch 1/3, Batch 130/269, Loss: 6.9052
Epoch 1/3, Batch 140/269, Loss: 6.9435
Epoch 1/3, Batch 150/269, Loss: 6.5489
Epoch 1/3, Batch 160/269, Loss: 7.3520
Epoch 1/3, Batch 170/269, Loss: 7.0133
Epoch 1/3, Batch 180/269, Loss: 6.8629
Epoch 1/3, Batch 190/269, Loss: 6.6532
Epoch 1/3, Batch 200/269, Loss: 6.3401
Epoch 1/3, Batch 210/269, Loss: 6.3353
Epoch 1/3, Batch

In [ ]:
from transformers import GPT2Tokenizer


# Load the trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPTModel(GPT_CONFIG_124M).to(device)
model.load_state_dict(torch.load("gpt_model.pth"))
model.eval()  # Set the model to evaluation mode

# Load the tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Define a function to generate text based on the model
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]  # Get the last context_size tokens

        # Get predictions from the model
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]  # Focus on the last time step
        probas = torch.softmax(logits, dim=-1)  # Apply softmax to get probabilities

        # Get the index of the word with the highest probability
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # Append the new token to the sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

# Example: Start with some seed text
start_test = time.time()
start_context = "AI can change"
encoded = tokenizer.encode(start_context)
encoded_tensor = torch.tensor(encoded).unsqueeze(0).to(device)  # Shape: (1, n_tokens)
test_time_sec = round(time.time() - start_test, 2)

# Generate text
max_new_tokens = 50
context_size = GPT_CONFIG_124M["context_length"]
generated_ids = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=max_new_tokens,
    context_size=context_size
)
test_time_sec = round(time.time() - start_test, 2)
# Decode the generated tokens into text
decoded_text = tokenizer.decode(generated_ids.squeeze(0).tolist())
print("Generated Text:\n", decoded_text)



Generated Text:
 AI can change.org/web.com/web/web/web. The most the same day of the same day of the same day of the same day of the same day of the same day of the same day of the same day of the same day


Experiment logged successfully!
